# Fitting the Phase Model

In the example `piprkit.PhaseModel` is used to fit a model to a measurement of the pupil. The model is split into a baseline (`BaselinePhase`), a constriction (`ConstrictionPhase`), a maintained constriction (`MaintainedConstrictionPhase`) and a redilation phase (`RedilationPhase`), and a line or an exponential is fitted to each.

A full description can be found in [API Reference](../../../api.rst).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import piprkit

pm = piprkit.load_simulated_pupil()

## Fitting the PhaseModel

The `PhaseModel` attempts to find a best fit throughout the full measurement. A variety of options are available to ensure the model converges to an optimal fit. These options will be covered later.

`get_formula_string` prints the formula for each section with parameters. A variety of plot functions is used to display the model: `plot` to show the fit line, `plot_sections` to display the sections (pre-stimulus, during-stimulus and post-stimulus), `plot_phases` to plot the phases themselves, and `plot_light_stimulus` to show the light stimulus.

In [ ]:
model = piprkit.PhaseModel(pm, baseline_duration=5)
print(model.get_formula_string())

fig, ax = plt.subplots(2, 1, figsize=(8, 5))
for axis in ax:
    model.plot(ax=axis, scatter=True, s=4, color="grey", label="data")
    model.plot_sections(ax=axis)
    model.plot_phases(ax=axis)
    model.plot_light_stimulus(ax=axis)
ax[1].set_xlim(-2, 3)
ax[0].legend(fontsize=7)
plt.show()

## Parameters, windows and flags

Using `get_all_params` the parameters of the fit can be retrieved that make up the formula obtained by `get_formula_string`, with `get_param` for a specific one. `get_window` retrieves the time window corresponding to the phase. `get_flags` gets any flags that might be set during the fitting procedure.

In [ ]:
for phase, params in model.get_all_params().items():
    print(f"{phase}: " + ", ".join(f"{name} = {value:.4f}" for name, value in params.items()))
print(f"redilation k: {model.get_param('redilation', 'k'):.4f}")
print()
for phase, window in model.get_window().items():
    print(f"{phase}: {window['start']:.3f} s to {window['end']:.3f} s, delay {window['delay']} s, {window['n_samples']} samples")
print()
for phase, flags in model.get_flags().items():
    print(f"{phase}: {flags}")
print()
print(model.get_options())

## Goodness of fit

The goodness-of-fit can be determined using `get_goodness_of_fit`. It compares the model with the actual data and, by default, uses MAE. Alternatively, `RMSE` and `R2` are available.

In [ ]:
for method in ["MAE", "RMSE", "R2"]:
    print(f"{method}: {model.get_goodness_of_fit(method):.4f}")
print()
for name, phase in model.get_phases().items():
    print(f"{name} RMSE: {phase.get_goodness_of_fit('RMSE'):.4f}")

## Options

A variety of options can be passed. See the [API](../../../api.rst) for full details. In the example, we use the phases option to select the baseline and redilation phase, and set the peak constriction window and redilation delay.

In [ ]:
options = [
    {},
    {"phases": ("baseline", "redilation"), "peak_constriction_window": 2.0, "redilation_delay": 0.5},
]

fig, ax = plt.subplots(len(options), 1, figsize=(8, 5), sharex=True)
for axis, kwargs in zip(ax, options):
    option_model = piprkit.PhaseModel(pm, baseline_duration=5, **kwargs)
    option_model.plot(ax=axis, scatter=True, s=4, color="grey", label="data")
    option_model.plot_phases(ax=axis)
    option_model.plot_light_stimulus(ax=axis)
    axis.set_title(", ".join(f"{name}={value}" for name, value in kwargs.items()) or "defaults", fontsize=9)
    axis.set_xlim(-2, 4)
    axis.legend(fontsize=7)
plt.show()

## Metrics from the fitted model

The model is a different type of object from a measurement, and has different options. In order to get a regular `PupilMeasurement` object, the `to_measurement` function can be used. This can be used to retrieve metrics as usual.

In [ ]:
fitted = model.to_measurement()
data_pipr = pm.metrics.pipr_6s()
fit_pipr = fitted.metrics.pipr_6s()
print(f"{data_pipr.get_name()} (data): {data_pipr:.2f}")
print(f"{fit_pipr.get_name()} (fit):  {fit_pipr:.2f}")

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
pm.plot(ax=ax, color="grey", label="data")
fitted.plot(ax=ax, color="black", linestyle="--", label="fit")
pm.plot_light_stimulus(ax=ax)
data_pipr.plot(ax=ax, color="blue", label=f"{data_pipr.get_label()} (data)")
fit_pipr.plot(ax=ax, color="red", label=f"{fit_pipr.get_label()} (fit)")
ax.legend(fontsize=7)
plt.show()

## Fitting a single phase

When solely interested in a section of the pupillary light response, individual phases can be modelled. Currently available are the `BaselinePhase`, `ConstrictionPhase`, `MaintainedConstrictionPhase`, and `RedilationPhase`. More details on fitting these phases can be found in the [API Reference](../../../api.rst), with an example given below.

In [ ]:
time_data, size_data = pm.get_time(), pm.get_size()
windows = model.get_window()

baseline = piprkit.BaselinePhase(time_data, size_data, windows["baseline"]["start"], windows["baseline"]["end"])
baseline.fit(float(np.nanmean(baseline.get_size())))

constriction = piprkit.ConstrictionPhase(
    time_data, size_data, windows["constriction"]["start"], windows["constriction"]["end"]
)
constriction.fit()

maintained_constriction = piprkit.MaintainedConstrictionPhase(
    time_data, size_data, windows["maintained_constriction"]["start"], windows["maintained_constriction"]["end"]
)
maintained_constriction.fit()

redilation = piprkit.RedilationPhase(time_data, size_data, windows["redilation"]["start"])
redilation.fit()

phases = [baseline, constriction, maintained_constriction, redilation]

fig, ax = plt.subplots(1, 1, figsize=(8, 3))
pm.plot(ax=ax, scatter=True, s=4, color="lightgrey", label="data")
for color, phase in zip(["C0", "C1", "C2", "C3"], phases):
    print(phase.get_formula_string())
    phase.plot_window(ax=ax, color=color)
    curve_time = np.linspace(phase.get_start_time(), phase.get_end_time(), 200)
    phase.plot_predicted(ax=ax, time_data=curve_time, color=color, label=type(phase).__name__)
ax.set_xlim(-2, 3)
ax.legend(fontsize=7)
plt.show()